# Classificador de Spam (NLP Básico)

Este notebook mostra, passo a passo, como treinar um classificador supervisionado para prever se uma mensagem é **spam** ou **ham** (não spam).

## O que você vai aprender
- Como carregar e entender o dataset `SMS Spam Collection`.
- Como limpar texto com NLTK.
- Como transformar texto em números com TF-IDF.
- Como treinar `MultinomialNB` e `LogisticRegression`.
- Como avaliar os modelos de forma legível.

> Objetivo didático: mostrar claramente os dados antes e depois do pré-processamento para análise humana antes do treino.

In [5]:
from pathlib import Path
import sys
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Resolve project root whether notebook is run from project root or notebooks/
cwd = Path.cwd().resolve()
if (cwd / "src").exists():
    PROJECT_ROOT = cwd
elif (cwd.parent / "src").exists():
    PROJECT_ROOT = cwd.parent
else:
    raise RuntimeError("Could not locate project root containing src/")

SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

from load_data import default_raw_path, download_sms_spam_collection, load_sms_spam
from preprocess import ensure_nltk_data, preprocess_series

pd.set_option("display.max_colwidth", 160)
PROJECT_ROOT

PosixPath('/home/marcos_maio/projects/ml_engineer_projects/basic_project/classify_messages_as_spam_or_not_spam')

## 1) Configuração dos hiperparâmetros

Nesta célula você controla os principais parâmetros do experimento. Como é um notebook didático, deixar tudo explícito ajuda a reproduzir resultados.

In [6]:
RANDOM_STATE = 42
TEST_SIZE = 0.20
MAX_FEATURES = 10_000
REMOVE_STOPWORDS = True

print({
    "RANDOM_STATE": RANDOM_STATE,
    "TEST_SIZE": TEST_SIZE,
    "MAX_FEATURES": MAX_FEATURES,
    "REMOVE_STOPWORDS": REMOVE_STOPWORDS,
})

{'RANDOM_STATE': 42, 'TEST_SIZE': 0.2, 'MAX_FEATURES': 10000, 'REMOVE_STOPWORDS': True}


## 2) Carregar dataset

Aqui carregamos o `SMS Spam Collection`. Se o arquivo não existir em `data/raw/`, ele é baixado automaticamente da UCI.

In [7]:
raw_path = default_raw_path()
if not raw_path.is_file():
    raw_path = download_sms_spam_collection()

df = load_sms_spam(raw_path)
print(f"Dataset path: {raw_path}")
print(f"Shape: {df.shape}")
df.head(10)

Dataset path: /home/marcos_maio/projects/ml_engineer_projects/basic_project/classify_messages_as_spam_or_not_spam/data/raw/SMSSpamCollection
Shape: (5572, 2)


,label,text
0,0,"Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat..."
1,0,Ok lar... Joking wif u oni...
2,1,Free entry in 2 a wkly comp to win FA Cup final tkts 21st May 2005. Text FA to 87121 to receive entry question(std txt rate)T&C's apply 08452810075over18's
3,0,U dun say so early hor... U c already then say...
4,0,"Nah I don't think he goes to usf, he lives around here though"
5,1,"FreeMsg Hey there darling it's been 3 week's now and no word back! I'd like some fun you up for it still? Tb ok! XxX std chgs to send, £1.50 to rcv"
6,0,Even my brother is not like to speak with me. They treat me like aids patent.
7,0,As per your request 'Melle Melle (Oru Minnaminunginte Nurungu Vettam)' has been set as your callertune for all Callers. Press *9 to copy your friends Caller...
8,1,WINNER!! As a valued network customer you have been selected to receivea £900 prize reward! To claim call 09061701461. Claim code KL341. Valid 12 hours only.
9,1,Had your mobile 11 months or more? U R entitled to Update to the latest colour mobiles with camera for Free! Call The Mobile Update Co FREE on 08002986030


## 3) Análise inicial do DataFrame (dados brutos)

Antes de treinar, analisamos os dados para entender:
- distribuição das classes (spam vs ham),
- tamanho médio das mensagens,
- qualidade geral dos textos.

In [8]:
analysis_df = df.copy()
analysis_df["text_length"] = analysis_df["text"].str.len()
analysis_df["token_count_raw"] = analysis_df["text"].str.split().str.len()

print("Class distribution (0=ham, 1=spam):")
display(analysis_df["label"].value_counts().to_frame("count"))

print("Length stats by class:")
display(
    analysis_df.groupby("label")["text_length"].describe()[["mean", "std", "min", "50%", "max"]]
)

print("Raw sample:")
display(analysis_df[["label", "text", "text_length", "token_count_raw"]].head(8))

Class distribution (0=ham, 1=spam):


,count
label,
0,4825
1,747


Length stats by class:


,mean,std,min,50%,max
label,,,,,
0,71.482487,58.440652,2.0,52.0,910.0
1,138.670683,28.873603,13.0,149.0,223.0


Raw sample:


,label,text,text_length,token_count_raw
0,0,"Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat...",111,20
1,0,Ok lar... Joking wif u oni...,29,6
2,1,Free entry in 2 a wkly comp to win FA Cup final tkts 21st May 2005. Text FA to 87121 to receive entry question(std txt rate)T&C's apply 08452810075over18's,155,28
3,0,U dun say so early hor... U c already then say...,49,11
4,0,"Nah I don't think he goes to usf, he lives around here though",61,13
5,1,"FreeMsg Hey there darling it's been 3 week's now and no word back! I'd like some fun you up for it still? Tb ok! XxX std chgs to send, £1.50 to rcv",147,32
6,0,Even my brother is not like to speak with me. They treat me like aids patent.,77,16
7,0,As per your request 'Melle Melle (Oru Minnaminunginte Nurungu Vettam)' has been set as your callertune for all Callers. Press *9 to copy your friends Caller...,160,26


## 4) Pré-processamento de texto (NLTK)

Aplicamos limpeza para padronizar as mensagens:
- lowercase,
- tokenização,
- remoção de ruído,
- lematização,
- remoção opcional de stopwords.

O objetivo é preparar os dados para o TF-IDF.

In [9]:
ensure_nltk_data()

analysis_df["text_clean"] = preprocess_series(
    analysis_df["text"].tolist(),
    remove_stopwords=REMOVE_STOPWORDS,
)
analysis_df["token_count_clean"] = analysis_df["text_clean"].str.split().str.len()

display(
    analysis_df[["label", "text", "text_clean", "text_length", "token_count_raw", "token_count_clean"]].head(12)
)

print("Exemplo de mensagens SPAM após limpeza:")
display(analysis_df.loc[analysis_df["label"] == 1, ["text", "text_clean"]].head(5))

print("Exemplo de mensagens HAM após limpeza:")
display(analysis_df.loc[analysis_df["label"] == 0, ["text", "text_clean"]].head(5))

,label,text,text_clean,text_length,token_count_raw,token_count_clean
0,0,"Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat...",go jurong point crazy available bugis n great world la e buffet cine got amore wat,111,20,16
1,0,Ok lar... Joking wif u oni...,ok lar joking wif u oni,29,6,6
2,1,Free entry in 2 a wkly comp to win FA Cup final tkts 21st May 2005. Text FA to 87121 to receive entry question(std txt rate)T&C's apply 08452810075over18's,free entry 2 wkly comp win fa cup final tkts 21st may 2005 text fa 87121 receive entry question std txt rate c apply 08452810075over18,155,28,25
3,0,U dun say so early hor... U c already then say...,u dun say early hor u c already say,49,11,9
4,0,"Nah I don't think he goes to usf, he lives around here though",nah nt think go usf life around though,61,13,8
5,1,"FreeMsg Hey there darling it's been 3 week's now and no word back! I'd like some fun you up for it still? Tb ok! XxX std chgs to send, £1.50 to rcv",freemsg hey darling 3 week word back like fun still tb ok xxx std chgs send 150 rcv,147,32,18
6,0,Even my brother is not like to speak with me. They treat me like aids patent.,even brother like speak treat like aid patent,77,16,8
7,0,As per your request 'Melle Melle (Oru Minnaminunginte Nurungu Vettam)' has been set as your callertune for all Callers. Press *9 to copy your friends Caller...,per request melle melle oru minnaminunginte nurungu vettam set callertune caller press 9 copy friend callertune,160,26,16
8,1,WINNER!! As a valued network customer you have been selected to receivea £900 prize reward! To claim call 09061701461. Claim code KL341. Valid 12 hours only.,winner valued network customer selected receivea 900 prize reward claim call 09061701461 claim code kl341 valid 12 hour,157,26,18
9,1,Had your mobile 11 months or more? U R entitled to Update to the latest colour mobiles with camera for Free! Call The Mobile Update Co FREE on 08002986030,mobile 11 month u r entitled update latest colour mobile camera free call mobile update co free 08002986030,154,29,18


Exemplo de mensagens SPAM após limpeza:


,text,text_clean
2,Free entry in 2 a wkly comp to win FA Cup final tkts 21st May 2005. Text FA to 87121 to receive entry question(std txt rate)T&C's apply 08452810075over18's,free entry 2 wkly comp win fa cup final tkts 21st may 2005 text fa 87121 receive entry question std txt rate c apply 08452810075over18
5,"FreeMsg Hey there darling it's been 3 week's now and no word back! I'd like some fun you up for it still? Tb ok! XxX std chgs to send, £1.50 to rcv",freemsg hey darling 3 week word back like fun still tb ok xxx std chgs send 150 rcv
8,WINNER!! As a valued network customer you have been selected to receivea £900 prize reward! To claim call 09061701461. Claim code KL341. Valid 12 hours only.,winner valued network customer selected receivea 900 prize reward claim call 09061701461 claim code kl341 valid 12 hour
9,Had your mobile 11 months or more? U R entitled to Update to the latest colour mobiles with camera for Free! Call The Mobile Update Co FREE on 08002986030,mobile 11 month u r entitled update latest colour mobile camera free call mobile update co free 08002986030
11,"SIX chances to win CASH! From 100 to 20,000 pounds txt> CSH11 and send to 87575. Cost 150p/day, 6days, 16+ TsandCs apply Reply HL 4 info",six chance win cash 100 20000 pound txt csh11 send 87575 cost 150pday 6days 16 tsandcs apply reply hl 4 info


Exemplo de mensagens HAM após limpeza:


,text,text_clean
0,"Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat...",go jurong point crazy available bugis n great world la e buffet cine got amore wat
1,Ok lar... Joking wif u oni...,ok lar joking wif u oni
3,U dun say so early hor... U c already then say...,u dun say early hor u c already say
4,"Nah I don't think he goes to usf, he lives around here though",nah nt think go usf life around though
6,Even my brother is not like to speak with me. They treat me like aids patent.,even brother like speak treat like aid patent


## 5) Divisão treino/teste

Agora separamos os dados em:
- **treino**: usado para aprender,
- **teste**: usado para validar desempenho em dados não vistos.

Usamos `stratify` para manter a proporção de classes nas duas partes.

In [ ]:
X = analysis_df["text_clean"].tolist()
y = analysis_df["label"].values

# X_train e X_test = entradas (features)
# y_train e y_test = saídas (labels)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)

split_df = pd.DataFrame(
    {
        "split": ["train", "test"],
        "samples": [len(X_train), len(X_test)],
        "spam_ratio": [float((y_train == 1).mean()), float((y_test == 1).mean())],
    }
)
display(split_df)

,split,samples,spam_ratio
0,train,4457,0.134171
1,test,1115,0.133632


## 6) Vetorização TF-IDF

Nesta etapa transformamos texto em números. Cada mensagem vira um vetor de pesos TF-IDF.

Configuração usada:
- `ngram_range=(1,2)` para unigramas e bigramas,
- `max_features` para limitar vocabulário,
- `min_df=2` para reduzir termos raríssimos.

In [11]:
vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    max_features=MAX_FEATURES,
    min_df=2,
    sublinear_tf=True,
)

X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

print("X_train_vec shape:", X_train_vec.shape)
print("X_test_vec shape:", X_test_vec.shape)

feature_names = vectorizer.get_feature_names_out()
preview_features = pd.DataFrame({"feature": feature_names[:25]})
preview_features

X_train_vec shape: (4457, 7664)
X_test_vec shape: (1115, 7664)


,feature
0,01223585334
1,01223585334 cum
2,020603
3,020603 2nd
4,0207
5,0207 153
6,02073162414
7,02073162414 cost
8,050703
9,050703 csbcm4235wc1n3xx


## 7) Treinar modelos: Naive Bayes e Regressão Logística

Treinamos dois classificadores clássicos para comparar:
- `MultinomialNB` (baseline rápido para texto)
- `LogisticRegression` (modelo linear forte em TF-IDF).

In [ ]:
nb = MultinomialNB(alpha=0.1)
nb.fit(X_train_vec, y_train)
y_pred_nb = nb.predict(X_test_vec)

lr = LogisticRegression(max_iter=2000, class_weight="balanced", solver="lbfgs")
lr.fit(X_train_vec, y_train)
y_pred_lr = lr.predict(X_test_vec)

print("Treino concluído.")

## 8) Avaliação comparativa (saída legível)

Aqui comparamos os dois modelos com:
- acurácia,
- precision/recall/F1,
- matriz de confusão.

Foco importante: **recall da classe spam** (quantos spams o modelo conseguiu capturar).

In [ ]:
metrics_table = pd.DataFrame(
    {
        "model": ["MultinomialNB", "LogisticRegression"],
        "accuracy": [
            accuracy_score(y_test, y_pred_nb),
            accuracy_score(y_test, y_pred_lr),
        ],
        "spam_recall": [
            classification_report(y_test, y_pred_nb, output_dict=True)["1"]["recall"],
            classification_report(y_test, y_pred_lr, output_dict=True)["1"]["recall"],
        ],
        "spam_precision": [
            classification_report(y_test, y_pred_nb, output_dict=True)["1"]["precision"],
            classification_report(y_test, y_pred_lr, output_dict=True)["1"]["precision"],
        ],
        "spam_f1": [
            classification_report(y_test, y_pred_nb, output_dict=True)["1"]["f1-score"],
            classification_report(y_test, y_pred_lr, output_dict=True)["1"]["f1-score"],
        ],
    }
).sort_values("spam_f1", ascending=False)

display(metrics_table)

print("Classification report - MultinomialNB")
print(classification_report(y_test, y_pred_nb, target_names=["ham", "spam"]))
print("Confusion matrix - MultinomialNB")
display(pd.DataFrame(confusion_matrix(y_test, y_pred_nb), index=["true_ham", "true_spam"], columns=["pred_ham", "pred_spam"]))

print("Classification report - LogisticRegression")
print(classification_report(y_test, y_pred_lr, target_names=["ham", "spam"]))
print("Confusion matrix - LogisticRegression")
display(pd.DataFrame(confusion_matrix(y_test, y_pred_lr), index=["true_ham", "true_spam"], columns=["pred_ham", "pred_spam"]))

## 9) (Opcional) Salvar artefatos para inferência

Se você quiser usar o `predict.py`, salve o mesmo vectorizer e modelos treinados no notebook.

In [ ]:
import joblib

models_dir = PROJECT_ROOT / "models"
models_dir.mkdir(parents=True, exist_ok=True)

joblib.dump(vectorizer, models_dir / "tfidf_vectorizer.joblib")
joblib.dump(nb, models_dir / "multinomial_nb.joblib")
joblib.dump(lr, models_dir / "logistic_regression.joblib")
joblib.dump({"remove_stopwords": REMOVE_STOPWORDS}, models_dir / "preprocess_config.joblib")

print(f"Artefatos salvos em: {models_dir}")

## 10) Seção opcional: migração para embeddings (BERT)

Quando você quiser avançar além de TF-IDF:

1. Instale dependências (`sentence-transformers`, `torch`).
2. Use o script já pronto `src/train_bert_extension.py` para comparar com o baseline.

No terminal:

```bash
python src/train_bert_extension.py
```

Ou apenas embeddings (sem reimprimir baseline):

```bash
python src/train_bert_extension.py --skip-tfidf-baseline
```

Essa etapa é mais custosa computacionalmente, mas permite capturar contexto semântico melhor que bag-of-words em muitos cenários.

## 11) Conclusão

Você executou um pipeline completo de NLP clássico supervisionado:

- dados brutos carregados e inspecionados,
- dados limpos e comparados com o original,
- treino e teste separados corretamente,
- vetorização TF-IDF,
- treino de dois modelos,
- avaliação comparativa com foco em qualidade para spam.

Próximo passo recomendado: testar pequenas variações de hiperparâmetros e comparar o impacto no `spam_recall` e `spam_f1`.